# EDA (Exploratory Data Analysis) of BDG dataset
## About BDG (Building Data Genome)

BDG (Building Data Genome) Dataset consists of several data points energy data spaning 2 years, here we are going to use this data to find anomalies in the points and conduct EDA (Exploratory Data Analysis) to find trends and get an idea of the dataset and do some cleaning.

## Loading Libraries

In [ ]:
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os

## Loading Data

In [ ]:
data_path = "C:/Users/Akshay/Desktop/building-energy-anomaly-detection/data/meters/raw/"

df_list = [] 

for file in os.listdir(data_path): 
    if file.endswith('.csv'): 
        df = pd.read_csv(os.path.join(data_path, file)) 
        df_list.append(df)
        df = pd.concat(df_list, ignore_index=True)

df.to_csv('eda.csv', index=False)

## EDA (Exploratory Data Analysis)
Now we conduct EDA on the joined dataset

In [ ]:
print(df.head())
print(df.info())
print(df.describe())
print(df.columns)

## Outcomes of EDA

We have found the size of the new joined dataset and have a glimpse of the data points and columns, each entry has a "timestamp" that follows YEAR-MONTH-DAY SECONDS-MINUTES-HOURS format, which requires cleaning. 

SIZE OF THE DATASET = 8 rows and 1600+ columns

## Visualizations

### 1. Time Series Plot - Energy Consumption Over Time

In [ ]:
# Select a few buildings for visualization
sample_buildings = df.columns[1:4]  # First 3 buildings after timestamp

plt.figure(figsize=(14, 6))
for building in sample_buildings:
    plt.plot(df['timestamp'][:168], df[building][:168], label=building, alpha=0.7)
plt.title('Energy Consumption Over Time (First Week)')
plt.xlabel('Timestamp')
plt.ylabel('Energy Consumption')
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### 2. Distribution Plot - Histogram of Energy Consumption

In [ ]:
# Flatten all building values for distribution plot
all_values = df.iloc[:, 1:].values.flatten()
all_values = all_values[~np.isnan(all_values)]  # Remove NaN values

plt.figure(figsize=(12, 6))
plt.hist(all_values, bins=50, color='skyblue', edgecolor='black', alpha=0.7)
plt.title('Distribution of Energy Consumption Values')
plt.xlabel('Energy Consumption')
plt.ylabel('Frequency')
plt.yscale('log')  # Log scale due to wide range of values
plt.tight_layout()
plt.show()

### 3. Box Plot - Energy Consumption by Building Type

In [ ]:
# Extract building types from column names
building_types = {}
for col in df.columns[1:]:
    if '_' in col:
        btype = col.split('_')[1]
        building_types[btype] = building_types.get(btype, []) + [col]

# Create box plot for top building types
top_types = list(building_types.keys())[:6]  # Top 6 building types
plt.figure(figsize=(14, 6))
box_data = []
labels = []
for btype in top_types:
    if building_types[btype]:
        values = df[building_types[btype]].values.flatten()
        values = values[~np.isnan(values)]
        if len(values) > 0:
            box_data.append(values)
            labels.append(btype)

plt.boxplot(box_data, labels=labels)
plt.title('Energy Consumption by Building Type')
plt.xlabel('Building Type')
plt.ylabel('Energy Consumption')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### 4. Heatmap - Correlation Between Buildings (Sample)

In [ ]:
# Select a subset of buildings for correlation
sample_cols = df.columns[1:21]  # First 20 buildings

plt.figure(figsize=(14, 10))
correlation_matrix = df[sample_cols].corr()
sns.heatmap(correlation_matrix, cmap='coolwarm', center=0, 
            annot=False, square=True, linewidths=0.5)
plt.title('Correlation Heatmap Between Buildings (Sample)')
plt.tight_layout()
plt.show()

### 5. Daily/Hourly Patterns - Average Consumption by Hour

In [ ]:
# Convert timestamp to datetime and extract hour
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['hour'] = df['timestamp'].dt.hour

# Calculate average consumption by hour
hourly_by_time = df.groupby('hour').mean()

plt.figure(figsize=(14, 6))
plt.plot(hourly_by_time.index, hourly_by_time.iloc[:, :-1].mean(axis=1), 
         marker='o', linewidth=2, color='green')
plt.title('Average Energy Consumption by Hour of Day')
plt.xlabel('Hour of Day')
plt.ylabel('Average Energy Consumption')
plt.xticks(range(24))
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Summary statistics for hourly pattern
print("\nHourly Pattern Summary:")
print("Peak Hour:", hourly_by_time.iloc[:, :-1].mean(axis=1).idxmax())
print("Lowest Hour:", hourly_by_time.iloc[:, :-1].mean(axis=1).idxmin())